# Step 4: Few-Shot + Chain-of-Thought Annotation

## Setup

In [77]:
from pathlib import Path
import pandas as pd
import numpy as np
import os
import importlib
import requests
import time
import krippendorff
from dotenv import load_dotenv
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    cohen_kappa_score,
    confusion_matrix
)

load_dotenv()
api_key = os.getenv("CAC_API_KEY")

HOST  = "https://maki.uni-mannheim.de/v1"
MODEL = "ministral-3-14b"

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

VALID_LABELS = ["NO_CRIME_FRAME", "CRIME_FRAME_NEG", "CRIME_FRAME_POS"]

print(f"API key loaded: {'YES' if api_key else 'NO'}")
print(f"Host: {HOST} | Model: {MODEL}")

API key loaded: YES
Host: https://maki.uni-mannheim.de/v1 | Model: ministral-3-14b


## Load Human Ground Truth and Annotation Setup

In [78]:
human_completed = pd.read_csv(RESULTS_DIR / "step3_testset_500_human_completed_translated.csv")

# use label_marmee as ground truth — overwrite existing final_human_label
human_completed["final_human_label"] = human_completed["label_marmee"]

human_completed = human_completed[
    human_completed["final_human_label"].isin(VALID_LABELS)
].copy().reset_index(drop=True)

print(f"Evaluation rows: {len(human_completed)}")
print(human_completed["final_human_label"].value_counts())

Evaluation rows: 250
final_human_label
CRIME_FRAME_NEG    136
NO_CRIME_FRAME     104
CRIME_FRAME_POS     10
Name: count, dtype: int64


In [79]:
# human_completed = pd.read_csv(RESULTS_DIR / "step3_testset_500_human_completed.csv")
# human_completed = human_completed[
#     human_completed["final_human_label"].isin(VALID_LABELS)
# ].copy().reset_index(drop=True)

# print(f"Evaluation rows: {len(human_completed)}")
# print(human_completed["final_human_label"].value_counts())

In [80]:
import annotation_setup
importlib.reload(annotation_setup)

from annotation_setup import (
    VALID_LABELS,
    instructions,
    reminder,
    parse_label,
    parse_explanation
)

print("Annotation setup loaded.")

Annotation setup loaded.


## Core API Function

In [81]:
def call_api(messages, temperature=0.0):
    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": messages
    }
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    for attempt in range(3):
        try:
            r = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if r.status_code == 200:
                return r.json()["choices"][0]["message"]["content"].strip()
            print(f"  [attempt {attempt+1}] status {r.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] error: {e}")
            time.sleep(5)
    return "ERROR"

print("API function loaded.")

API function loaded.


## Few-Shot Examples

Eight invented examples targeting the four systematic error patterns from Step 3.
These do not appear in the Step 3 test set.

Pattern coverage:
- Example 1: clear NO_CRIME_FRAME (integration context)
- Example 2: clear CRIME_FRAME_NEG (direct perpetrator)
- Example 3: CRIME_FRAME_NEG via institutional security framing (fixes miss error)
- Example 4: NO_CRIME_FRAME — vague opinion framing is not enough (fixes overcall error)
- Example 5: NO_CRIME_FRAME — group as victim of violence
- Example 6: CRIME_FRAME_POS — successful security operation (fixes low recall)
- Example 7: CRIME_FRAME_NEG — group legitimizes terrorism without being direct perpetrator
- Example 8: NO_CRIME_FRAME — international war reporting between states

In [82]:
FEW_SHOT_EXAMPLES = [
    # 1. Clear NO_CRIME_FRAME — integration context, no security link
    {
        "text": "Die Lage in der Unterkunft hat sich nach Angaben der Sozialarbeiter entspannt. Viele [Gruppe 1] haetten inzwischen Deutschkurse begonnen und bemuehten sich aktiv um eine Arbeitsstelle. Die Caritas lobte das Engagement der Freiwilligen, die regelmaessig Unterstuetzung anbieten.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Der Absatz beschreibt Integration und ehrenamtliche Hilfe ohne jeden Bezug zu Kriminalitaet, Sicherheit oder Praevention."
    },
    # 2. Clear CRIME_FRAME_NEG — direct perpetrator
    {
        "text": "Ein 34-jaehriger [Gruppe 1] wurde am Freitagabend am Hauptbahnhof festgenommen. Er soll in den vergangenen Wochen mehrfach Passanten mit einem Messer bedroht und Handtaschen entrissen haben. Die Polizei hat ihn als Serientaeter eingestuft.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Ein Gruppenmitglied wird explizit als Taeter schwerer Straftaten dargestellt und direkt mit Kriminalitaet verknuepft."
    },
    # 3. CRIME_FRAME_NEG — institutional security framing (fixes missed cases)
    {
        "text": "Die Innenministerkonferenz hat beschlossen, die erkennungsdienstliche Erfassung aller einreisenden [Gruppe 1] zu verschaerfen. Innenminister Hoffmann erklaerte, man koenne sich keine Sicherheitsluecken leisten. Es muesse sichergestellt werden, dass keine Schleuser, Kriminellen oder Extremisten die Situation ausnutzten.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Auch ohne konkrete Straftat wird die Gruppe institutionell als potenzielle Sicherheitsgefahr gerahmt — die explizite Verknuepfung mit Kriminalitaet und Extremismus durch eine Regierungsstimme ist CRIME_FRAME_NEG."
    },
    # 4. NO_CRIME_FRAME — group is victim of violence
    {
        "text": "Auf dem Gelaende der Fluechtlingsunterkunft in Tempelhof wurden in der Nacht Fensterscheiben eingeworfen und Hakenkreuze gesprueht. Die Polizei ermittelt wegen schwerer Sachbeschaedigung und Volksverhetzung. Die [Gruppe 1] Bewohner wurden unverletzt, aber stark veraengstigt in eine Notunterkunft gebracht.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Die [Gruppe 1] sind Opfer eines rechtsextremen Angriffs — eine Gruppe als Opfer von Gewalt darzustellen ist kein Sicherheitsrahmen gegen diese Gruppe."
    },
    # location not mentioned and no Germany link = NO_CRIME_FRAME
    {
        "text": "Berichte zufolge wurden mehrere [Gruppe 1] bei Auseinandersetzungen verletzt. Wo genau die Vorfaelle stattfanden, blieb unklar. Ein Bezug zu Deutschland oder deutschen Sicherheitsbehoerden wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kein expliziter Bezug zu Deutschland oder zur deutschen inneren Sicherheit — ohne diesen Bezug gilt NO_CRIME_FRAME."
    },
    # 5. NO_CRIME_FRAME — international war reporting between states
    {
        "text": "Die [Gruppe 1] Streitkraefte haben nach Angaben des Verteidigungsministeriums in Kiew erneut mehrere Staedte im Osten beschossen. Bei den Angriffen kamen nach ukrainischen Angaben mindestens zwoelf Zivilisten ums Leben. Die NATO beriet ueber weitere Waffenlieferungen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Kriegsberichterstattung ueber einen Konflikt zwischen Staaten ohne Bezug zu Minderheiten in Deutschland erhaelt immer NO_CRIME_FRAME."
    },
    # Germany-link exception — foreign crime but explicit Germany connection = CRIME_FRAME_NEG
    {
        "text": "Der Attentaeter, der in Wien drei Menschen toetete, hatte mehrere Jahre in Deutschland gelebt und war deutschen Behoerden als Gefaehrder bekannt. Das Bundesinnenministerium prueft nun, ob Sicherheitsluecken im deutschen System vorlagen.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Obwohl die Tat in Oesterreich stattfand, wird ein expliziter Bezug zur deutschen Sicherheit hergestellt — der Taeter lebte in Deutschland und deutsche Behoerden sind beteiligt. Das ergibt CRIME_FRAME_NEG."
    },
    # 6. CRIME_FRAME_NEG — group legitimizes terrorism without being direct perpetrator
    {
        "text": "Sprecher der [Gruppe 1] bezeichneten den Anschlag als legitimen Widerstand und erklaerten, die Opfer seien selbst schuld. In sozialen Medien verbreiteten Anhaenger der Gruppe Videos, in denen der Terrorakt gefeiert wurde.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe legitimiert und feiert explizit einen Terroranschlag — das zaehlt als CRIME_FRAME_NEG auch ohne direkte Taeterschaft."
    },
    # 7. CRIME_FRAME_POS — successful security operation
    {
        "text": "Das Gemeinsame Terrorismusabwehrzentrum hat nach intensiver Zusammenarbeit von Polizei und Geheimdiensten einen islamistischen Anschlag in Muenchen vereitelt. Drei Verdaechtige wurden festgenommen. Innenministerin Weber sprach von einem grossen Erfolg der Sicherheitsbehoerden.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Die Sicherheitsbehoerden werden explizit als erfolgreiche Akteure gerahmt die eine Bedrohung abgewendet haben — Schutz und Sicherheitsgewinn stehen im Vordergrund."
    },
    # 8. CRIME_FRAME_POS — social factors explanation + differentiation (aligns with Robin's definition)
    {
        "text": "Experten erklaeren den Anstieg der Jugendkriminalitaet in Brennpunktvierteln vor allem mit sozialer Ausgrenzung und fehlenden Perspektiven fuer [Gruppe 1] Jugendliche. Kriminologin Mueller betonte, man duerfe keinen Generalverdacht gegen die Gruppe hegen — die grosse Mehrheit halte sich an Gesetze.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Kriminalitaet wird auf soziale Ursachen zurueckgefuehrt und eine explizite Differenzierung wird vorgenommen — das ist CRIME_FRAME_POS nach der Praventions- und Differenzierungsdefinition."
    },
    # 9. CRIME_FRAME_POS — prevention/deradicalization program
    {
        "text": "Das neue Deradikalisierungsprogramm der Bundesregierung soll vor allem [Gruppe 1] Jugendliche ansprechen, die gefaehrdet sind, in extremistische Kreise abzudriften. Sozialarbeiter begleiten die Teilnehmer ueber zwei Jahre und helfen beim Einstieg in Ausbildung und Arbeit.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Ein explizites Praeventionsprogramm im Sicherheitskontext — Schutz und Praevention stehen im Vordergrund, nicht Taeterschaft."
    },
    # 10. NO_CRIME_FRAME — integration without security frame (boundary with CRIME_FRAME_POS)
    {
        "text": "Die Stadtratsfraktion diskutierte Massnahmen zur besseren Integration von [Gruppe 1]. Themen waren Sprachkurse, Wohnraumvermittlung und Beschaeftigungsfoerderung. Ein konkreter Sicherheits- oder Kriminalitaetsbezug wurde nicht hergestellt.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Integration ohne jeden Sicherheits- oder Kriminalitaetsbezug ist NO_CRIME_FRAME — CRIME_FRAME_POS erfordert einen expliziten Sicherheits- oder Praeventionsrahmen."
    },
    # fixes id 14, 21, 135 — generalised threat/disorder framing
    {
        "text": "Die Behörden berichten, dass sich in Teilen der Stadt regelrechte Parallelgesellschaften gebildet haben, in denen [Gruppe 1] unter sich bleiben und staatliche Kontrolle kaum noch greift. Anwohner klagen über Verwahrlosung und mangelnde Sicherheit.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird mit Kontrollverlust, Unordnung und mangelnder Sicherheit im deutschen Inland verknüpft — das ist generalisierte gesellschaftliche Bedrohungsrahmung und zählt als CRIME_FRAME_NEG."
    },

    # fixes id 30 — explicit domestic security risk framing
    {
        "text": "Der Botschafter warnte: 'Diese [Gruppe 1] stellen ein Sicherheitsrisiko für Deutschland dar.' Es sei falsch, sie aufzunehmen, da sie in deutschen Städten für Unruhe sorgen würden.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird von einer öffentlichen Person explizit als Sicherheitsrisiko für Deutschland bezeichnet — das ist CRIME_FRAME_NEG unabhängig davon ob ein konkretes Verbrechen genannt wird."
    },
    # fixes id 15, 47, 113, 123, 189 — political commentary overcalling
    {
        "text": "Die Politikerin forderte eine bessere Integrationspolitik für [Gruppe 1]. In der Debatte wurden auch Fragen der öffentlichen Ordnung angesprochen, ohne jedoch konkrete Vorfälle oder Straftaten zu nennen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Allgemeine politische Debatten über Integration oder öffentliche Ordnung ohne explizite Verknüpfung der Gruppe mit konkreten Straftaten, Verdacht oder Ermittlungen sind NO_CRIME_FRAME."
    }
]

print(f"Few-shot examples loaded: {len(FEW_SHOT_EXAMPLES)}")
for i, ex in enumerate(FEW_SHOT_EXAMPLES):
    print(f"  {i+1}. {ex['label']}")

Few-shot examples loaded: 15
  1. NO_CRIME_FRAME
  2. CRIME_FRAME_NEG
  3. CRIME_FRAME_NEG
  4. NO_CRIME_FRAME
  5. NO_CRIME_FRAME
  6. NO_CRIME_FRAME
  7. CRIME_FRAME_NEG
  8. CRIME_FRAME_NEG
  9. CRIME_FRAME_POS
  10. CRIME_FRAME_POS
  11. CRIME_FRAME_POS
  12. NO_CRIME_FRAME
  13. CRIME_FRAME_NEG
  14. CRIME_FRAME_NEG
  15. NO_CRIME_FRAME


## Annotation Functions

In [83]:
def format_few_shot_block(examples, include_explanation=True):
    blocks = []
    for ex in examples:
        block = f"Text:\n{ex['text']}\n\nLabel: {ex['label']}"
        if include_explanation:
            block += f"\nExplanation: {ex['explanation']}"
        blocks.append(block)
    return "\n\n---\n\n".join(blocks)


def annotate_few_shot(text, temperature=0.0):
    few_shot_block = format_few_shot_block(FEW_SHOT_EXAMPLES, include_explanation=False)
    prompt = (
        f"{instructions}\n\n"
        f"Hier sind Beispiele fuer die Annotation:\n\n{few_shot_block}\n\n---\n\n"
        f"Jetzt annotieren Sie diesen Absatz:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>\n"
        "Explanation: <ein Satz>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen genau."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, temperature)


def annotate_few_shot_cot(text, temperature=0.0):
    few_shot_block = format_few_shot_block(FEW_SHOT_EXAMPLES, include_explanation=True)
    prompt = (
        f"{instructions}\n\n"
        f"Hier sind Beispiele mit Begruendungen:\n\n{few_shot_block}\n\n---\n\n"
        f"Beantworten Sie vor der Annotation diese vier Fragen:\n"
        f"1. Ist die genannte Gruppe Taeter oder Opfer?\n"
        f"2. Handelt es sich um Kriegsberichterstattung zwischen Staaten ohne Bezug zu Minderheiten in Deutschland?\n"
        f"3. Wird eine Sicherheitsmassnahme als expliziter Erfolg gerahmt?\n"
        f"4. Wird die Gruppe explizit als Quelle von Bedrohung oder Kriminalitaet dargestellt, oder legitimiert sie Gewalt?\n\n"
        f"Text:\n\n{text}\n\n"
        f"{reminder}\n\n"
        "Antworten Sie genau in diesem Format:\n"
        "Reasoning: <kurze Antworten auf die vier Fragen>\n"
        "Label: <NO_CRIME_FRAME oder CRIME_FRAME_NEG oder CRIME_FRAME_POS>\n"
        "Explanation: <ein Satz>"
    )
    messages = [
        {"role": "system", "content": "Sie sind ein Experte fuer Inhaltsanalyse. Beantworten Sie zuerst die Kontrollfragen, dann vergeben Sie das Label."},
        {"role": "user",   "content": prompt}
    ]
    return call_api(messages, temperature)


def parse_reasoning(raw):
    for line in raw.split("\n"):
        if line.lower().startswith("reasoning:"):
            return line[len("reasoning:"):].strip()
    return ""


print("Annotation functions loaded.")

Annotation functions loaded.


## Condition B: Few-Shot Only

In [84]:
OUTPUT_B = RESULTS_DIR / "step4_fewshot_results.csv"

if OUTPUT_B.exists():
    done_b = pd.read_csv(OUTPUT_B)
    done_ids_b = set(done_b["testset_id"])
    print(f"Resuming: {len(done_b)} done, {len(human_completed) - len(done_b)} remaining.")
else:
    done_b = pd.DataFrame()
    done_ids_b = set()
    print(f"Starting fresh: {len(human_completed)} rows.")

buffer_b = []

for i, row in human_completed.iterrows():
    if row["testset_id"] in done_ids_b:
        continue

    raw         = annotate_few_shot(str(row["text_block_en"]))
    label       = parse_label(raw)
    explanation = parse_explanation(raw)

    buffer_b.append({
        "testset_id":          row["testset_id"],
        "group":               row["group"],
        "text_block_en":       row["text_block_en"],
        "final_human_label":   row["final_human_label"],
        "fewshot_label":       label,
        "fewshot_explanation": explanation,
        "fewshot_raw":         raw,
    })
    done_ids_b.add(row["testset_id"])

    if len(buffer_b) % 50 == 0:
        chunk = pd.DataFrame(buffer_b)
        chunk.to_csv(OUTPUT_B, mode="a", header=not OUTPUT_B.exists(), index=False)
        done_b  = pd.concat([done_b, chunk], ignore_index=True)
        buffer_b = []
        print(f"  [{len(done_b)}/{len(human_completed)}] checkpointed")

if buffer_b:
    chunk = pd.DataFrame(buffer_b)
    chunk.to_csv(OUTPUT_B, mode="a", header=not OUTPUT_B.exists(), index=False)
    done_b = pd.concat([done_b, chunk], ignore_index=True)

print(f"\nCondition B done. {len(done_b)} rows.")
print(done_b["fewshot_label"].value_counts())

Resuming: 174 done, 76 remaining.
  [224/250] checkpointed

Condition B done. 250 rows.
fewshot_label
NO_CRIME_FRAME     148
CRIME_FRAME_NEG    100
CRIME_FRAME_POS      2
Name: count, dtype: int64


## Condition C: Few-Shot + Chain-of-Thought

In [85]:
OUTPUT_C = RESULTS_DIR / "step4_fewshot_cot_results.csv"

if OUTPUT_C.exists():
    done_c = pd.read_csv(OUTPUT_C)
    done_ids_c = set(done_c["testset_id"])
    print(f"Resuming: {len(done_c)} done, {len(human_completed) - len(done_c)} remaining.")
else:
    done_c = pd.DataFrame()
    done_ids_c = set()
    print(f"Starting fresh: {len(human_completed)} rows.")

buffer_c = []

for i, row in human_completed.iterrows():
    if row["testset_id"] in done_ids_c:
        continue

    raw         = annotate_few_shot_cot(str(row["text_block_en"]))
    label       = parse_label(raw)
    explanation = parse_explanation(raw)
    reasoning   = parse_reasoning(raw)

    buffer_c.append({
        "testset_id":        row["testset_id"],
        "group":             row["group"],
        "text_block_en":     row["text_block_en"],
        "final_human_label": row["final_human_label"],
        "cot_label":         label,
        "cot_reasoning":     reasoning,
        "cot_explanation":   explanation,
        "cot_raw":           raw,
    })
    done_ids_c.add(row["testset_id"])

    if len(buffer_c) % 50 == 0:
        chunk = pd.DataFrame(buffer_c)
        chunk.to_csv(OUTPUT_C, mode="a", header=not OUTPUT_C.exists(), index=False)
        done_c  = pd.concat([done_c, chunk], ignore_index=True)
        buffer_c = []
        print(f"  [{len(done_c)}/{len(human_completed)}] checkpointed")

if buffer_c:
    chunk = pd.DataFrame(buffer_c)
    chunk.to_csv(OUTPUT_C, mode="a", header=not OUTPUT_C.exists(), index=False)
    done_c = pd.concat([done_c, chunk], ignore_index=True)

print(f"\nCondition C done. {len(done_c)} rows.")
print(done_c["cot_label"].value_counts())

Resuming: 174 done, 76 remaining.
  [224/250] checkpointed

Condition C done. 250 rows.
cot_label
NO_CRIME_FRAME     125
CRIME_FRAME_NEG    123
CRIME_FRAME_POS      2
Name: count, dtype: int64


## Evaluation

In [86]:
def evaluate(y_true, y_pred, condition_name):
    valid = [
        (t, p) for t, p in zip(y_true, y_pred)
        if t in VALID_LABELS and p in VALID_LABELS
    ]
    if not valid:
        print(f"{condition_name}: no valid predictions")
        return {}

    yt, yp   = zip(*valid)
    acc      = accuracy_score(yt, yp)
    f1_macro = f1_score(yt, yp, average="macro", labels=VALID_LABELS, zero_division=0)
    f1_w     = f1_score(yt, yp, average="weighted", labels=VALID_LABELS, zero_division=0)
    kappa    = cohen_kappa_score(yt, yp, labels=VALID_LABELS)

    alpha_data = np.array([
        [VALID_LABELS.index(t) for t in yt],
        [VALID_LABELS.index(p) for p in yp]
    ])
    try:
        alpha = krippendorff.alpha(alpha_data, level_of_measurement="nominal")
    except Exception:
        alpha = float("nan")

    print(f"\n{'='*50}")
    print(f"Condition: {condition_name}")
    print(f"{'='*50}")
    print(f"Rows evaluated:        {len(yt)}")
    print(f"Accuracy:              {acc:.3f}")
    print(f"Macro F1:              {f1_macro:.3f}")
    print(f"Weighted F1:           {f1_w:.3f}")
    print(f"Cohen's kappa:         {kappa:.3f}")
    print(f"Krippendorff's alpha:  {alpha:.3f}")
    print()
    print(classification_report(yt, yp, labels=VALID_LABELS, zero_division=0))
    print("Confusion matrix (rows=human, cols=model):")
    print(confusion_matrix(yt, yp, labels=VALID_LABELS))
    print(f"Label order: {VALID_LABELS}")

    return {
        "condition":    condition_name,
        "n":            len(yt),
        "accuracy":     acc,
        "macro_f1":     f1_macro,
        "weighted_f1":  f1_w,
        "kappa":        kappa,
        "alpha":        alpha,
    }

print("Evaluation function loaded.")

Evaluation function loaded.


In [87]:
# Condition A: zero-shot baseline from Step 3
model_info = pd.read_csv(RESULTS_DIR / "step3_testset_500_internal_with_model_info.csv")
baseline = human_completed.merge(
    model_info[["testset_id", "model_label"]],
    on="testset_id",
    how="left"
)
results_a = evaluate(
    baseline["final_human_label"].tolist(),
    baseline["model_label"].tolist(),
    "A: Zero-shot baseline (Step 3)"
)


Condition: A: Zero-shot baseline (Step 3)
Rows evaluated:        250
Accuracy:              0.644
Macro F1:              0.492
Weighted F1:           0.635
Cohen's kappa:         0.304
Krippendorff's alpha:  0.304

                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.60      0.60      0.60       104
CRIME_FRAME_NEG       0.68      0.72      0.70       136
CRIME_FRAME_POS       1.00      0.10      0.18        10

       accuracy                           0.64       250
      macro avg       0.76      0.47      0.49       250
   weighted avg       0.66      0.64      0.63       250

Confusion matrix (rows=human, cols=model):
[[62 42  0]
 [38 98  0]
 [ 4  5  1]]
Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']


In [88]:
done_b = pd.read_csv(OUTPUT_B)
results_b = evaluate(
    done_b["final_human_label"].tolist(),
    done_b["fewshot_label"].tolist(),
    "B: Few-shot"
)


Condition: B: Few-shot
Rows evaluated:        250
Accuracy:              0.668
Macro F1:              0.509
Weighted F1:           0.659
Cohen's kappa:         0.380
Krippendorff's alpha:  0.366

                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.58      0.83      0.68       104
CRIME_FRAME_NEG       0.80      0.59      0.68       136
CRIME_FRAME_POS       0.50      0.10      0.17        10

       accuracy                           0.67       250
      macro avg       0.63      0.51      0.51       250
   weighted avg       0.70      0.67      0.66       250

Confusion matrix (rows=human, cols=model):
[[86 17  1]
 [56 80  0]
 [ 6  3  1]]
Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']


In [89]:
done_c = pd.read_csv(OUTPUT_C)
results_c = evaluate(
    done_c["final_human_label"].tolist(),
    done_c["cot_label"].tolist(),
    "C: Few-shot + CoT"
)


Condition: C: Few-shot + CoT
Rows evaluated:        250
Accuracy:              0.660
Macro F1:              0.502
Weighted F1:           0.654
Cohen's kappa:         0.351
Krippendorff's alpha:  0.349

                 precision    recall  f1-score   support

 NO_CRIME_FRAME       0.58      0.70      0.64       104
CRIME_FRAME_NEG       0.74      0.67      0.70       136
CRIME_FRAME_POS       0.50      0.10      0.17        10

       accuracy                           0.66       250
      macro avg       0.61      0.49      0.50       250
   weighted avg       0.67      0.66      0.65       250

Confusion matrix (rows=human, cols=model):
[[73 30  1]
 [45 91  0]
 [ 7  2  1]]
Label order: ['NO_CRIME_FRAME', 'CRIME_FRAME_NEG', 'CRIME_FRAME_POS']


In [90]:
summary = pd.DataFrame([r for r in [results_a, results_b, results_c] if r])
summary = summary.set_index("condition").round(3)

print("\n=== Summary: All Conditions ===")
print(summary.to_string())

summary.to_csv(RESULTS_DIR / "step4_evaluation_summary.csv")
print("\nSaved to results/step4_evaluation_summary.csv")


=== Summary: All Conditions ===
                                  n  accuracy  macro_f1  weighted_f1  kappa  alpha
condition                                                                         
A: Zero-shot baseline (Step 3)  250     0.644     0.492        0.635  0.304  0.304
B: Few-shot                     250     0.668     0.509        0.659  0.380  0.366
C: Few-shot + CoT               250     0.660     0.502        0.654  0.351  0.349

Saved to results/step4_evaluation_summary.csv


## Error Analysis

Inspect remaining errors in the best condition to understand what is still hard and whether the few-shot examples resolved the three main error patterns.

In [91]:
# change best_col to 'fewshot_label' if condition B performed better
best     = done_c.copy()
best_col = "cot_label"

best["correct"] = best["final_human_label"] == best[best_col]
errors = best[~best["correct"]].copy()

print(f"Remaining errors: {len(errors)} / {len(best)}")
print(errors.groupby(["final_human_label", best_col]).size().reset_index(name="count"))

Remaining errors: 85 / 250
  final_human_label        cot_label  count
0   CRIME_FRAME_NEG   NO_CRIME_FRAME     45
1   CRIME_FRAME_POS  CRIME_FRAME_NEG      2
2   CRIME_FRAME_POS   NO_CRIME_FRAME      7
3    NO_CRIME_FRAME  CRIME_FRAME_NEG     30
4    NO_CRIME_FRAME  CRIME_FRAME_POS      1


In [92]:
for (human_lbl, model_lbl), group in errors.groupby(["final_human_label", best_col]):
    print(f"\n{'='*60}")
    print(f"Human: {human_lbl}  |  Model: {model_lbl}  |  Count: {len(group)}")
    print(f"{'='*60}")
    for _, row in group.head(5).iterrows():
        print(f"\nGroup: {row['group']}")
        print(f"Text: {str(row['text_block_en'])[:400]}")
        if "cot_reasoning" in row.index:
            print(f"Reasoning: {row['cot_reasoning']}")
        print()


Human: CRIME_FRAME_NEG  |  Model: NO_CRIME_FRAME  |  Count: 45

Group: nan
Text: Die Einwohner des Stadtteils Chloraka monieren, dort habe sich eine Art Getto gebildet, in dem Hunderte [Gruppe 1] und [Gruppe 1] lebten und die Polizei die Kontrolle verloren habe, berichteten zyprische Medien übereinstimmend.

Die EU-Inselrepublik Zypern bittet immer wieder die anderen EU-Mitgliedstaaten darum, auf der Insel angekommene Schutzsuchende aufzunehmen. Allein im vergangenen Oktober u
Reasoning: nan


Group: ISLMST
Text: In Ägypten sind drei Journalisten nach rund eineinhalb Jahren Haft freigelassen worden. Diaa Raschwan vom ägyptischen Journalistenverband veröffentlichte am Sonntag Fotos von Ammer Abdel-Moneim, Hani Greischa und Essam Abdin in weißer Häftlingskleidung. Sie umarmten ihre Familienangehörigen. Die drei Journalisten waren wegen separater Vorwürfe in Untersuchungshaft gewesen. Ihnen wurde missbräuchli
Reasoning: nan


Group: REFUG
Text: DRESDEN. Die tschechisch-deutsche Grenze is

In [93]:
# CRIME_FRAME_POS improvement check
pos_human = best[best["final_human_label"] == "CRIME_FRAME_POS"]
print(f"CRIME_FRAME_POS human cases: {len(pos_human)}")
print(f"Correctly labelled: {(pos_human[best_col] == 'CRIME_FRAME_POS').sum()}")
print(f"Step 3 baseline correctly labelled: 1 (recall 14%)")
print()
print(pos_human[["group", "final_human_label", best_col, "cot_reasoning", "text_block_en"]].to_string())

CRIME_FRAME_POS human cases: 10
Correctly labelled: 1
Step 3 baseline correctly labelled: 1 (recall 14%)

      group final_human_label        cot_label  cot_reasoning                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

## Results Table for Presentation

Fill in after running all conditions:

| Condition | Macro F1 | Kappa | NEG F1 | NO_CRIME F1 | POS F1 |
|-----------|----------|-------|--------|-------------|--------|
| A: Zero-shot | 0.521 | 0.321 | 0.70 | 0.61 | 0.25 |
| B: Few-shot | | | | | |
| C: Few-shot + CoT | | | | | |

In [94]:
done_b = pd.read_csv(OUTPUT_B)

misses = done_b[
    (done_b["final_human_label"] == "CRIME_FRAME_NEG") &
    (done_b["fewshot_label"] == "NO_CRIME_FRAME")
].copy().reset_index(drop=True)

print(f"CRIME_FRAME_NEG missed as NO_CRIME_FRAME: {len(misses)}\n")

for i, row in misses.iterrows():
    print(f"{'='*60}")
    print(f"#{i+1} | testset_id: {row['testset_id']} | group: {row['group']}")
    print(f"Human label:  {row['final_human_label']}")
    print(f"Model label:  {row['fewshot_label']}")
    print(f"Explanation:  {row['fewshot_explanation']}")
    print(f"Text: {str(row['text_block_en'])}")
    print()

CRIME_FRAME_NEG missed as NO_CRIME_FRAME: 56

#1 | testset_id: 14 | group: nan
Human label:  CRIME_FRAME_NEG
Model label:  NO_CRIME_FRAME
Explanation:  Der Text erwähnt zwar die Bildung eines „Gettos“ und den Verlust von Polizeikontrolle, verknüpft dies jedoch nicht explizit mit Kriminalität, Straftaten, Sicherheitsbedrohungen oder konkreten Vorfällen der Gruppe – es handelt sich um eine allgemeine, nicht spezifizierte Kritik an sozialer Segregation. Zudem findet die Handlung außerhalb Deutschlands statt, ohne Bezug zur deutschen inneren Sicherheit.
Text: Die Einwohner des Stadtteils Chloraka monieren, dort habe sich eine Art Getto gebildet, in dem Hunderte [Gruppe 1] und [Gruppe 1] lebten und die Polizei die Kontrolle verloren habe, berichteten zyprische Medien übereinstimmend.

Die EU-Inselrepublik Zypern bittet immer wieder die anderen EU-Mitgliedstaaten darum, auf der Insel angekommene Schutzsuchende aufzunehmen. Allein im vergangenen Oktober und November sind auf Zypern nach Angab

In [95]:
all_errors = done_b[
    done_b["final_human_label"] != done_b["fewshot_label"]
].copy().reset_index(drop=True)

print(f"Total errors in condition B: {len(all_errors)} / {len(done_b)}\n")
print("Error pattern summary:")
print(all_errors.groupby(["final_human_label", "fewshot_label"]).size().reset_index(name="count").to_string(index=False))

print(f"\n{'='*60}")
print("ALL ERRORS — human label | model label | group | text")
print(f"{'='*60}\n")

for i, row in all_errors.iterrows():
    print(f"#{i+1} | testset_id: {row['testset_id']}")
    print(f"  Human:  {row['final_human_label']}")
    print(f"  Model:  {row['fewshot_label']}")
    print(f"  Group:  {row['group']}")
    print(f"  Explanation: {row['fewshot_explanation']}")
    print(f"  Text: {str(row['text_block_en'])[:350]}")
    print()

Total errors in condition B: 83 / 250

Error pattern summary:
final_human_label   fewshot_label  count
  CRIME_FRAME_NEG  NO_CRIME_FRAME     56
  CRIME_FRAME_POS CRIME_FRAME_NEG      3
  CRIME_FRAME_POS  NO_CRIME_FRAME      6
   NO_CRIME_FRAME CRIME_FRAME_NEG     17
   NO_CRIME_FRAME CRIME_FRAME_POS      1

ALL ERRORS — human label | model label | group | text

#1 | testset_id: 14
  Human:  CRIME_FRAME_NEG
  Model:  NO_CRIME_FRAME
  Group:  nan
  Explanation: Der Text erwähnt zwar die Bildung eines „Gettos“ und den Verlust von Polizeikontrolle, verknüpft dies jedoch nicht explizit mit Kriminalität, Straftaten, Sicherheitsbedrohungen oder konkreten Vorfällen der Gruppe – es handelt sich um eine allgemeine, nicht spezifizierte Kritik an sozialer Segregation. Zudem findet die Handlung außerhalb Deutschlands statt, ohne Bezug zur deutschen inneren Sicherheit.
  Text: Die Einwohner des Stadtteils Chloraka monieren, dort habe sich eine Art Getto gebildet, in dem Hunderte [Gruppe 1] und [Grup

In [96]:
# Cases where you+Robin agree but model disagrees
# Split into two groups:
# 1. You both say CRIME_FRAME_NEG but model says NO_CRIME_FRAME
# 2. You both say NO_CRIME_FRAME but model says CRIME_FRAME_NEG

human_annotation = pd.read_csv(RESULTS_DIR / "step3_testset_500_for_human_annotation.csv")
done_b           = pd.read_csv(OUTPUT_B)
human_completed  = pd.read_csv(RESULTS_DIR / "step3_testset_500_human_completed_translated.csv")
human_completed  = human_completed.rename(columns={"marmee_label": "final_human_label"})

# get all errors where both annotators agreed
errors_b = done_b[done_b["final_human_label"] != done_b["fewshot_label"]].copy()

errors_detail = errors_b.merge(
    human_annotation[["testset_id", "label_robin", "label_marmee"]],
    on="testset_id",
    how="left"
)

both_agree_errors = errors_detail[
    errors_detail["label_robin"] == errors_detail["label_marmee"]
].copy()

# split into the two error directions
missed_neg = both_agree_errors[
    (both_agree_errors["final_human_label"] == "CRIME_FRAME_NEG") &
    (both_agree_errors["fewshot_label"] == "NO_CRIME_FRAME")
].copy()
missed_neg["error_type"] = "human=CRIME_FRAME_NEG model=NO_CRIME_FRAME"

overcalled = both_agree_errors[
    (both_agree_errors["final_human_label"] == "NO_CRIME_FRAME") &
    (both_agree_errors["fewshot_label"] == "CRIME_FRAME_NEG")
].copy()
overcalled["error_type"] = "human=NO_CRIME_FRAME model=CRIME_FRAME_NEG"

relabel_candidates = pd.concat([missed_neg, overcalled], ignore_index=True)

# add agreed_label column for you and Robin to fill in
relabel_candidates["agreed_label"] = ""

# reorder columns
cols_first = [
    "testset_id", "group", "error_type",
    "text_block_en",
    "label_robin", "label_marmee",
    "fewshot_label", "fewshot_explanation",
    "agreed_label",
]
cols_rest = [c for c in relabel_candidates.columns if c not in cols_first]
relabel_candidates = relabel_candidates[cols_first + cols_rest]

output_path = RESULTS_DIR / "step4_relabelling_candidates.csv"
relabel_candidates.to_csv(output_path, index=False)

print(f"Saved {len(relabel_candidates)} rows")
print(f"  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME: {len(missed_neg)}")
print(f"  human=NO_CRIME_FRAME model=CRIME_FRAME_NEG: {len(overcalled)}")
print()
print(relabel_candidates[["testset_id", "group", "error_type", "label_robin", "fewshot_label", "agreed_label"]].to_string())

Saved 46 rows
  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME: 37
  human=NO_CRIME_FRAME model=CRIME_FRAME_NEG: 9

    testset_id   group                                  error_type      label_robin    fewshot_label agreed_label
0           14     NaN  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME  CRIME_FRAME_NEG   NO_CRIME_FRAME             
1           21   REFUG  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME  CRIME_FRAME_NEG   NO_CRIME_FRAME             
2           30     RUS  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME  CRIME_FRAME_NEG   NO_CRIME_FRAME             
3           32    JUDE  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME  CRIME_FRAME_NEG   NO_CRIME_FRAME             
4           44    MIGR  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME  CRIME_FRAME_NEG   NO_CRIME_FRAME             
5           50   REFUG  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME  CRIME_FRAME_NEG   NO_CRIME_FRAME             
6           55     UKR  human=CRIME_FRAME_NEG model=NO_CRIME_FRAME  CRIME_FRAME_NEG   NO_C